# EDA 심화 1 · 직접 풀어보는 연습문제 정답지

이 노트북은 `EDA_1_Overview_ECommerce_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 이커머스 분석을 위한 사전 데이터를 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np

ecommerce_df = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1010],
    "customer_id": ["C01", "C02", "C01", "C03", "C04", "C02", "C05", "C06", "C07", "C08", "C08"],
    "category": ["Fashion", "Tech", "Fashion", "Beauty", "Living", "Tech", "Fashion", "Living", "Tech", "Beauty", "Beauty"],
    "price": [45000, 320000, 28000, 18000, 65000, 1250000, 52000, 48000, 890000, 22000, 22000],
    "qty": [2, 1, 3, 2, 1, 1, 1, 2, 1, 3, 3],
    "delivery_days": [2, 3, 2, 1, 4, 8, 2, 3, 9, 1, 1],
    "rating": [4.5, 4.0, 4.8, 4.2, 3.8, 1.5, 4.6, 4.1, 2.0, 4.9, 4.9]
})
ecommerce_df["revenue"] = ecommerce_df["price"] * ecommerce_df["qty"]


## 문제 1. 이커머스 주문 데이터 무결성 검증 (결측치 및 중복 행 점검) [데이터 정제]

`ecommerce_df` 데이터프레임의 무결성을 점검하세요:
1. `ecommerce_df.shape`를 출력하세요.
2. 각 컬럼별 결측치 개수(`isna().sum()`)를 확인하세요.
3. 완전히 동일한 중복 행(`duplicated()`)이 존재하는지 확인하고, 중복 행을 제거한 `clean_orders` 데이터프레임을 생성하세요.
4. 중복 제거 후 남은 행의 개수를 출력하세요.


In [ ]:
# 문제 1 풀이
print("1. 원본 데이터 크기 (행, 열):", ecommerce_df.shape)

# 2. 결측치 확인
print("\n2. 컬럼별 결측치 건수:")
print(ecommerce_df.isna().sum())

# 3. 중복 행 확인 및 제거
dup_count = ecommerce_df.duplicated().sum()
print(f"\n3. 중복 행 개수: {dup_count}개")
clean_orders = ecommerce_df.drop_duplicates()

# 4. 정제 후 크기
print(f"4. 중복 제거 후 데이터 크기: {clean_orders.shape} (기존 {len(ecommerce_df)}행 -> {len(clean_orders)}행)")


**결과 해석 / 확인한 점:**  
1. **중복 데이터 유입 방지:** 1010번 주문이 2건 중복 등록되어 있어 이를 방치할 경우 매출이 이중 집계되는 오류가 발생합니다. `drop_duplicates()`로 총 11행에서 10행으로 정제했습니다.
2. **결측 없음 확인:** 모든 컬럼의 결측치가 0건으로 완벽한 데이터셋임을 확인했습니다.


## 문제 2. 결제 금액(revenue)의 기술 통계량과 IQR 기반 이상치 탐지 [이상치 분석]

`clean_orders['revenue']` 컬럼에 대해:
1. `describe()`를 출력하여 평균, 중앙값(50%), 최솟값, 최댓값을 확인하세요.
2. 1사분위수($Q1$)와 3사분위수($Q3$), $IQR$을 계산하세요.
3. 상한 한계선($Q3 + 1.5 \times IQR$)을 넘어서는 극단적 고액 결제 주문 건을 찾아 출력하세요.


In [ ]:
# 문제 2 풀이
rev = clean_orders["revenue"]

# 1. 기술 통계량
print("=== 결제 금액 기술 통계량 ===")
print(rev.describe().round(1))

# 2. IQR 계산
q1 = rev.quantile(0.25)
q3 = rev.quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr

print(f"\nQ1: {q1:,.0f}원 | Q3: {q3:,.0f}원 | IQR: {iqr:,.0f}원")
print(f"이상치 상한 기준액: {upper_limit:,.0f}원")

# 3. 이상치 행 추출
extreme_orders = clean_orders[clean_orders["revenue"] > upper_limit]
print("\n=== 상한 초과 고액 결제 이상치 목록 ===")
print(extreme_orders[["order_id", "customer_id", "category", "revenue"]])


**결과 해석 / 확인한 점:**  
1. **평균과 중앙값의 괴리:** 중앙값(87,000원)에 비해 평균(294,900원)이 약 3.4배 높은 것은 소수의 Tech 카테고리 초고액 주문(1,250,000원, 890,000원)이 평균을 강하게 끌어올렸기 때문입니다.
2. **IQR 기준치 검증:** 상한액인 562,125원(Q1=65,250원, Q3=264,000원, IQR=198,750원)을 초과하는 1006번과 1009번 Tech 주문이 통계적 이상치로 식별되었습니다.


## 문제 3. 카테고리별(category) 총 매출, 주문 건수 및 객단가 집계 [그룹 집계]

`clean_orders` 데이터에서 카테고리별 실적을 비교 분석하세요:
1. 카테고리별 총 매출액(`sum`), 주문 건수(`count`), 평균 결제액(`mean`)을 구하세요 (`groupby()`).
2. 각 카테고리의 총 매출액이 전체 매출에서 차지하는 매출 비중(%)을 계산하여 컬럼으로 추가하세요.
3. 총 매출액 기준 내림차순 정렬하여 출력하세요.


In [ ]:
# 문제 3 풀이
cat_summary = clean_orders.groupby("category")["revenue"].agg(
    총매출액="sum",
    주문건수="count",
    평균객단가="mean"
).reset_index()

total_revenue = cat_summary["총매출액"].sum()
cat_summary["매출비중(%)"] = (cat_summary["총매출액"] / total_revenue * 100).round(1)

# 총매출액 기준 내림차순 정렬
cat_summary = cat_summary.sort_values(by="총매출액", ascending=False)

print("=== 카테고리별 매출 실적 요약 ===")
print(cat_summary.to_string(index=False))


**결과 해석 / 확인한 점:**  
1. **파레토 법칙의 실증:** Tech 카테고리는 주문 건수가 3건에 불과하지만 전체 매출의 83.4%를 차지하며 비즈니스 매출을 견인하고 있습니다.
2. **객단가 격차:** Tech의 평균 객단가(820,000원)는 Fashion(75,333원)이나 Beauty(51,000원) 대비 압도적으로 높아 카테고리별 마케팅 전략이 달라져야 함을 시사합니다.


## 문제 4. 배송 소요 일수(delivery_days)와 고객 평점(rating)의 상관관계 [상관 분석]

물류 배송 속도가 고객 만족도에 미치는 영향을 검증하려고 합니다:
1. 배송 일수(`delivery_days`)와 고객 평점(`rating`) 간의 피어슨 상관계수(`.corr()`)를 구하세요.
2. 배송 일수가 3일 이하인 '빠른 배송' 그룹과 4일 이상인 '지연 배송' 그룹으로 분할하여 각 그룹의 평균 고객 평점을 비교하세요.


In [ ]:
# 문제 4 풀이
# 1. 상관계수 계산
corr_val = clean_orders["delivery_days"].corr(clean_orders["rating"])
print(f"배송 일수와 고객 평점 간 피어슨 상관계수: {corr_val:.4f}")

# 2. 그룹 분할 및 평균 평점 비교
fast_delivery = clean_orders[clean_orders["delivery_days"] <= 3]
delayed_delivery = clean_orders[clean_orders["delivery_days"] >= 4]

print(f"\n빠른 배송 (3일 이하) 평균 평점: {fast_delivery['rating'].mean():.2f}점 (표본 {len(fast_delivery)}건)")
print(f"지연 배송 (4일 이상) 평균 평점: {delayed_delivery['rating'].mean():.2f}점 (표본 {len(delayed_delivery)}건)")


**결과 해석 / 확인한 점:**  
1. **강한 음의 상관관계(r ≈ -0.96):** 상관계수가 -0.9569(약 -0.96)에 달해 배송 기간이 길어질수록 고객 만족도가 급격히 하락하는 극명한 음의 선형 관계가 확인되었습니다.
2. **비즈니스 액션:** 빠른 배송 그룹은 평균 4.44점의 높은 평점을 기록한 반면, 지연 배송(4일 이상)은 평균 2.43점으로 추락하므로 배송 지연 방지가 고객 유지의 핵심 과제임을 알 수 있습니다.


## 문제 5. 고객별 구매 빈도 및 충성 고객(VIP) 식별 [고객 분석]

`clean_orders` 데이터에서 고객 ID(`customer_id`)를 기준으로:
1. 고객별 총 주문 횟수(`frequency`)와 총 구매 금액(`monetary`)을 집계하세요.
2. 2회 이상 주문한 '재구매 고객'들의 목록을 출력하세요.
3. 가장 많은 매출을 기여한 1위 고객의 ID와 총 기여 매출액을 도출하세요.


In [ ]:
# 문제 5 풀이
customer_rfm = clean_orders.groupby("customer_id").agg(
    frequency=("order_id", "count"),
    monetary=("revenue", "sum")
).reset_index()

# 2회 이상 재구매 고객 필터링
repeat_buyers = customer_rfm[customer_rfm["frequency"] >= 2]
print("=== 재구매 고객 목록 (2회 이상) ===")
print(repeat_buyers.to_string(index=False))

# 1위 최고 기여 고객
top_customer = customer_rfm.sort_values(by="monetary", ascending=False).iloc[0]
print(f"\n최고 매출 기여 1위 고객: {top_customer['customer_id']} (총 매출: {top_customer['monetary']:,}원)")


**결과 해석 / 확인한 점:**  
1. **재구매 패턴:** C01(2회, 174,000원)과 C02(2회, 1,570,000원) 2명의 고객이 다회 구매를 기록했습니다.
2. **핵심 VIP 판별:** C02 고객은 단 2건의 주문으로 157만 원의 매출을 창출한 최고 핵심 VIP로 확인되었습니다.


## 문제 6. 종합 비즈니스 의사결정 인사이트 보고서 도출 [종합 인사이트]

지금까지 수행한 EDA 결과를 종합하여, 경영진에게 보고할 3가지 핵심 지표 요약과 권고사항을 딕셔너리 형태로 구조화하여 출력하세요:
- `key_findings`: 매출 쏠림, 배송 만족도, 재구매 기여도 등 3대 핵심 팩트
- `action_plan`: 물류 최적화 및 VIP 케어를 위한 실행 방안


In [ ]:
# 문제 6 풀이
eda_report = {
    "report_title": "이커머스 Q4 주문 데이터 EDA 및 전략 권고안",
    "key_findings": [
        "1. Tech 카테고리가 전체 매출의 83.4%를 점유하며 소수 고액 품목에 매출 집중",
        "2. 배송 소요 일수와 평점 간 상관계수 -0.9569(약 -0.96)로 4일 이상 배송 시 평점 급락(2.43점)",
        "3. 전체 고객 중 재구매율은 25%이며, VIP(C02) 1인이 전체 매출의 50% 이상(53.2%)을 기여"
    ],
    "action_plan": [
        "물류 개선: Tech 고가 상품의 3일 이내 보장 배송 풀필먼트 전담 라인 신설",
        "CRM 강화: 상위 20% 매출 기여 고객 대상 VIP 전담 케어 프로그램 도입"
    ]
}

print(f"[{eda_report['report_title']}]\n")
print("■ 핵심 발견 사항 (Key Findings):")
for f in eda_report["key_findings"]:
    print(f"  {f}")
print("\n■ 실행 권고사항 (Action Plans):")
for a in eda_report["action_plan"]:
    print(f"  {a}")

**결과 해석 / 확인한 점:**  
1. **EDA의 종착점:** 단순 계산을 넘어 데이터가 가리키는 현상을 명확한 비즈니스 언어로 번역하고 구체적인 실행 조치(배송 풀필먼트 최적화, VIP CRM)를 도출하는 EDA 본연의 목적을 완수했습니다.
